# Power Outage Analysis

**Name(s)**: Alina Gao, Fei Liang

**Website Link**: (your website link)

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

#from dsc80_utils import openpyxl # Feel free to uncomment and use this.

In [ ]:
# pip install openpyxl

## Step 1: Introduction

This project analyzes a dataset of major power outages in the United States from 2000 to 2016. The dataset contains 1534 outage events with 56 variables, including outage duration, cause category, climate region, and the number of customers affected.

Our research question is: Does the cause of a power outage (specifically severe weather versus other causes) significantly affect how long the outage lasts?

Power outages affect millions of people every year. When the power goes out, hospitals lose electricity, food spoils, and people lose heat or air conditioning. Knowing what causes the longest outages can help power companies prepare better and fix problems faster. Since storms and extreme weather are becoming more common, it is especially important to understand whether weather-related outages tend to last longer than outages from other causes.

## Step 2: Data Cleaning and Exploratory Data Analysis

In [6]:
df = pd.read_excel('outage.xlsx', skiprows=5, header=0)
df = df.drop(0).reset_index(drop=True)

print(df.shape)
df.head()

(1534, 57)


,variables,OBS,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,...,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
0,NaN,1.0,2011.0,7.0,Minnesota,MN,MRO,East North Central,-0.3,normal,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
1,NaN,2.0,2014.0,5.0,Minnesota,MN,MRO,East North Central,-0.1,normal,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
2,NaN,3.0,2010.0,10.0,Minnesota,MN,MRO,East North Central,-1.5,cold,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
3,NaN,4.0,2012.0,6.0,Minnesota,MN,MRO,East North Central,-0.1,normal,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
4,NaN,5.0,2015.0,7.0,Minnesota,MN,MRO,East North Central,1.2,warm,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743


In [7]:
df = df.drop(columns='variables')

print(df.shape)
df.head()

(1534, 56)


,OBS,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,...,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
0,1.0,2011.0,7.0,Minnesota,MN,MRO,East North Central,-0.3,normal,2011-07-01 00:00:00,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
1,2.0,2014.0,5.0,Minnesota,MN,MRO,East North Central,-0.1,normal,2014-05-11 00:00:00,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
2,3.0,2010.0,10.0,Minnesota,MN,MRO,East North Central,-1.5,cold,2010-10-26 00:00:00,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
3,4.0,2012.0,6.0,Minnesota,MN,MRO,East North Central,-0.1,normal,2012-06-19 00:00:00,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
4,5.0,2015.0,7.0,Minnesota,MN,MRO,East North Central,1.2,warm,2015-07-18 00:00:00,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743


In [8]:
fig = px.histogram(
    df,
    x='CAUSE.CATEGORY',
    title='Number of Power Outages by Cause Category',
    labels={'CAUSE.CATEGORY': 'Cause Category', 'count': 'Count'},
    color='CAUSE.CATEGORY'
)
fig.update_layout(showlegend=False)
fig.show()

## Step 3: Assessment of Missingness

**NMAR (Not Missing At Random) Analysis**<br>
Maybe some utility company negligence or liability-inducing equipment failure, the company may 
intentionally leave the detailed cause blank.
 

## Step 4: Hypothesis Testing

Null Hypothesis : The distribution of outage duration for severe weather events is the same as for non-severe-weather.

Alternative Hypothesis: Power outages caused by severe weather tend to last longer than those caused by other factors.

Test Statistic: Difference in means of outage duration based on weather condition.


In [13]:
df_ht = df[['CAUSE.CATEGORY', 'OUTAGE.DURATION']].dropna()
df_ht['is_weather'] = df_ht['CAUSE.CATEGORY'] == 'severe weather'

observed_diff = (
    df_ht.groupby('is_weather')['OUTAGE.DURATION'].mean()[True]
    - df_ht.groupby('is_weather')['OUTAGE.DURATION'].mean()[False]
)
print(f"Observed difference in means: {observed_diff:.2f} minutes")

n_repetitions = 1000
simulated_diffs = []

for _ in range(n_repetitions):
    shuffled = df_ht['is_weather'].sample(frac=1).reset_index(drop=True)
    temp = df_ht.copy()
    temp['is_weather'] = shuffled
    diff = (
        temp.groupby('is_weather')['OUTAGE.DURATION'].mean()[True]
        - temp.groupby('is_weather')['OUTAGE.DURATION'].mean()[False]
    )
    simulated_diffs.append(diff)

p_value = np.mean(np.array(simulated_diffs) >= observed_diff)
print(f"P-value: {p_value}")
# the p-vlaue is 0.0 so we reject the null hypothesis


Observed difference in means: 2537.81 minutes
P-value: 0.0


## Step 5: Framing a Prediction Problem

We plan to predict the column CAUSE.CATEGORY using features such as climate region, state, month, anomaly level, and number of customers affected. This is a classification problem since CAUSE.CATEGORY contains discrete categories.

## Step 6: Baseline Model

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

In [11]:
df_model= df[['CLIMATE.REGION', 'MONTH', 'CAUSE.CATEGORY']].dropna()

le= LabelEncoder()
df_model= df_model.copy()
df_model['CLIMATE.REGION']= le.fit_transform(df_model['CLIMATE.REGION'])

X= df_model[['CLIMATE.REGION', 'MONTH']]
y= df_model['CAUSE.CATEGORY']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline= DecisionTreeClassifier(max_depth=2, random_state=42)
baseline.fit(X_train, y_train)

y_pred= baseline.predict(X_test)
print(f"Baseline Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Baseline Accuracy: 0.4375


Our baseline model is a Decision Tree classifer with a maximum depth of 2, trained to predict the cause category of a power outage using only two features (climate region and month). We kept this baseline model simple intentionally as a starting point so we have a reference to compare against later when we make improvements. The baseline accuracy is like a benchmark, meaning our final model should perform better than this to be considered worthwhile. We plan to add more features such as state, anomaly level, number of customers affected, and outage duration. We will also increase the tree depth or switch to a better, more powerful algorithm.

## Step 7: Final Model

In [ ]:
# TODO

: 

## Step 8: Fairness Analysis

In [ ]:
# TODO

: 